# 대청댐 조류경보 예측 모델 학습 노트북

이 노트북은 **모델링 파트 전용 코드**입니다.

전처리팀이 만든 최종 모델 입력 테이블을 받아서 Gradient Boosting 기반 모델을 학습, 평가, 저장, 추론하는 흐름만 포함합니다.

포함하는 것:
- 최종 모델 입력 테이블 로드
- feature / target 분리
- 시간 기준 train / validation split
- Gradient Boosting 회귀·분류 모델 학습
- 평가 지표 계산 코드
- 모델 저장 / 로드 코드
- 추론 함수
- 조류경보 운영 보조 로직
- SHAP 또는 feature importance 기반 설명 함수

포함하지 않는 것:
- 원천 데이터 전처리
- 결측치 처리
- 이상치 처리
- 데이터 병합
- target 생성
- lag / rolling / 누적강수 / 변화량 / 계절성 / hydro-topology feature 생성
- Granger Causality / VAR 분석
- UI, API, DB, 배포 코드

실제 결과와 성능 수치는 데이터를 실행한 뒤 확인해야 합니다.


## 1. 입력 데이터 가정

이 노트북은 아래 파일이 이미 준비되어 있다고 가정합니다.

```text
data/processed/model_input/algae_model_input.csv
```

이 파일에는 이미 다음 작업이 끝나 있어야 합니다.

- 수질·기상·댐 운영 데이터 정리 및 병합
- 결측치/이상치 처리
- lag feature 생성
- rolling feature 생성
- 누적 강수량 feature 생성
- 변화량 feature 생성
- 계절성 feature 생성
- hydro-topology feature 생성
- target label 생성

따라서 이 노트북에서는 새로운 feature를 만들지 않습니다.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)


## 2. Config

컬럼명은 전처리팀 산출물에 맞게 여기서만 수정합니다.

모델링 코드 안에서 feature를 새로 만들지 않기 위해, feature column은 `DROP_COLUMNS`를 제외한 나머지 컬럼으로 구성합니다.


In [ ]:
# ============================================================
# Path Config
# ============================================================
MODEL_INPUT_PATH = Path("data/processed/model_input/algae_model_input.csv")
SCHEMA_PATH = Path("data/processed/model_input/model_input_schema.csv")  # optional
OUTPUT_DIR = Path("artifacts/models")
PREDICTION_DIR = Path("artifacts/predictions")
METRIC_DIR = Path("artifacts/metrics")
EXPLAIN_DIR = Path("artifacts/explain")

# ============================================================
# Column Config
# ============================================================
DATE_COLUMN = "sample_date"      # TODO: 전처리팀 산출물에 맞게 수정
SITE_COLUMN = "site"             # TODO: 전처리팀 산출물에 맞게 수정
SPLIT_COLUMN = "split"           # optional: 있으면 train/valid split에 우선 사용

REGRESSION_TARGET = "target_log_cells_t_plus_7"  # TODO: 실제 회귀 target 컬럼명으로 수정
CLASSIFICATION_TARGET = "target_alert_t_plus_7"  # TODO: 실제 분류 target 컬럼명으로 수정

# 모델 입력에서 제외할 컬럼입니다.
# target, 식별자, 날짜, 운영 라벨 등은 feature에서 제외합니다.
DROP_COLUMNS = [
    DATE_COLUMN,
    SITE_COLUMN,
    SPLIT_COLUMN,
    REGRESSION_TARGET,
    CLASSIFICATION_TARGET,
    "target_cells_t_plus_7",       # optional
    "alert_stage",                 # optional
    "raw_cells",                   # optional: 현재값인지 미래값인지 불명확하면 기본 제외
]

# ============================================================
# Feature Safety Config
# ============================================================
# 가장 안전한 방식은 전처리팀이 model_input_schema.csv에 role=feature로 승인한 컬럼만 사용하는 것입니다.
# schema 파일이 없으면 DROP_COLUMNS 기반으로 feature 후보를 만들고, 누수 의심 키워드를 자동 검사합니다.
SCHEMA_COLUMN_NAME = "column_name"
SCHEMA_ROLE_NAME = "role"
FEATURE_ROLE_VALUE = "feature"

FORBIDDEN_FEATURE_KEYWORDS = [
    "target",
    "future",
    "next",
    "t_plus",
    "label",
    "answer",
    "ground_truth",
]

# 문자열/categorical feature는 전처리팀이 미리 인코딩했다고 가정합니다.
# 따라서 모델링 노트북에서는 feature가 numeric인지 검사만 합니다.
REQUIRE_NUMERIC_FEATURES = True

# ============================================================
# Modeling Config
# ============================================================
RANDOM_STATE = 42
VALID_SIZE_RATIO = 0.2
PROBABILITY_THRESHOLD = 0.5  # TODO: validation 기준으로 조정 가능
ALERT_CELL_THRESHOLD = 1000  # TODO: 공식 기준 확인 후 config에서 확정

# NOTE:
# 아래 복원식은 target이 log10(cells + 1)로 생성된 경우에만 사용합니다.
# target 변환 방식이 다르면 inverse transform 함수를 수정해야 합니다.
REGRESSION_TARGET_IS_LOG10_CELLS_PLUS_1 = True


## 3. 최종 모델 입력 테이블 로드

여기서는 최종 모델 입력 테이블을 그대로 읽습니다.

전처리, 결측치 처리, feature engineering은 수행하지 않습니다.


In [ ]:
def load_model_input(path: Path) -> pd.DataFrame:
    # Load preprocessed model input table.
    # This function assumes preprocessing and feature engineering are already done.
    return pd.read_csv(path)


# 실제 실행 시 주석을 해제해서 사용합니다.
# df = load_model_input(MODEL_INPUT_PATH)
# df.head()


## 4. Feature / Target 분리

이 단계는 모델 학습 전에 **어떤 컬럼을 입력 feature로 사용할지** 정하는 부분입니다.

가장 중요한 목적은 **데이터 누수(Data Leakage) 방지**입니다. 모델이 예측 시점에 알 수 없는 정답 컬럼, 미래 컬럼, target 관련 컬럼을 입력으로 사용하면 검증 성능은 좋아 보일 수 있지만 실제 운영에서는 사용할 수 없습니다.

### 기본 원칙

1. 전처리팀이 `model_input_schema.csv`를 제공하면 `role == "feature"`로 승인된 컬럼만 사용합니다.
2. schema 파일이 없으면 `DROP_COLUMNS`에 포함된 컬럼을 제외한 나머지를 feature 후보로 사용합니다.
3. feature 후보에 `target`, `future`, `next`, `t_plus`, `label` 같은 누수 의심 키워드가 포함되면 에러를 발생시킵니다.
4. 이 노트북은 전처리를 담당하지 않으므로 문자열/categorical feature는 이미 숫자로 인코딩되어 있어야 합니다.

### schema 사용을 권장하는 이유

`DROP_COLUMNS` 방식은 편하지만, 새 target 컬럼이나 미래 정보 컬럼이 추가되었을 때 실수로 feature에 포함될 수 있습니다. 반면 schema 기반 whitelist는 전처리팀이 명시적으로 `feature`라고 승인한 컬럼만 사용하므로 가장 안전합니다.

권장 schema 예시:

```text
column_name,role,available_at_prediction,description
rain_7d_sum,feature,yes,최근 7일 누적강수량
residence_proxy,feature,yes,체류시간 proxy
target_alert_t_plus_7,target,no,7일 뒤 관심 이상 여부
target_log_cells_t_plus_7,target,no,7일 뒤 로그 세포수
sample_date,id,yes,기준 날짜
site,id,yes,지점명
split,meta,yes,학습/검증 구분
```

### feature에서 제외해야 하는 대표 컬럼

| 컬럼 유형 | 제외 이유 |
| --- | --- |
| 날짜/지점 식별자 | 모델 입력보다는 정렬, 그룹, 결과 표시 용도 |
| `split` | train/valid 구분용 관리 컬럼 |
| `target_*` | 정답 컬럼이므로 포함 시 데이터 누수 발생 |
| `future`, `next`, `t_plus` 포함 컬럼 | 예측 시점에 알 수 없는 미래 정보일 가능성 |
| 운영 라벨/발령 결과 | target 시점 정보일 수 있어 정의 확인 전 기본 제외 |

즉, 이 섹션의 함수들은 단순히 컬럼을 고르는 코드가 아니라, **모델이 정답지를 보고 학습하는 상황을 막기 위한 안전장치**입니다.


In [ ]:
def load_model_input_schema(schema_path: Path) -> pd.DataFrame | None:
    """Load optional model input schema.

    Schema is not feature engineering. It is a safety contract between preprocessing and modeling teams.
    """
    if not schema_path.exists():
        return None
    return pd.read_csv(schema_path)


def get_feature_columns_from_schema(
    df: pd.DataFrame,
    schema_df: pd.DataFrame,
    column_name_col: str = "column_name",
    role_col: str = "role",
    feature_role_value: str = "feature",
) -> list[str]:
    required_cols = {column_name_col, role_col}
    missing = required_cols - set(schema_df.columns)
    if missing:
        raise KeyError(f"Schema file is missing required columns: {sorted(missing)}")

    feature_columns = (
        schema_df.loc[schema_df[role_col].eq(feature_role_value), column_name_col]
        .dropna()
        .astype(str)
        .tolist()
    )

    missing_in_df = [col for col in feature_columns if col not in df.columns]
    if missing_in_df:
        raise KeyError(f"Schema-approved feature columns are missing in model input: {missing_in_df}")

    return feature_columns


def get_feature_columns_by_drop_rule(df: pd.DataFrame, drop_columns: list[str]) -> list[str]:
    existing_drop_columns = [col for col in drop_columns if col in df.columns]
    feature_columns = [col for col in df.columns if col not in existing_drop_columns]
    return feature_columns


def check_leakage_columns(
    feature_columns: list[str],
    forbidden_keywords: list[str],
) -> None:
    suspicious = [
        col for col in feature_columns
        if any(keyword in col.lower() for keyword in forbidden_keywords)
    ]
    if suspicious:
        raise ValueError(
            "누수 의심 컬럼이 feature에 포함되어 있습니다. "
            f"schema 또는 DROP_COLUMNS를 확인하세요: {suspicious}"
        )


def check_required_targets(
    df: pd.DataFrame,
    regression_target: str,
    classification_target: str,
) -> None:
    missing = [col for col in [regression_target, classification_target] if col not in df.columns]
    if missing:
        raise KeyError(f"Target columns are missing in model input: {missing}")


def check_numeric_features(df: pd.DataFrame, feature_columns: list[str]) -> None:
    non_numeric = [col for col in feature_columns if not pd.api.types.is_numeric_dtype(df[col])]
    if non_numeric:
        raise TypeError(
            "모델링 노트북은 전처리 완료 feature table을 입력으로 받습니다. "
            "문자형/categorical feature는 전처리팀에서 encoding 후 전달해야 합니다. "
            f"Non-numeric feature columns: {non_numeric}"
        )


def get_feature_columns(
    df: pd.DataFrame,
    drop_columns: list[str],
    schema_path: Path | None = None,
) -> list[str]:
    schema_df = load_model_input_schema(schema_path) if schema_path is not None else None

    if schema_df is not None:
        feature_columns = get_feature_columns_from_schema(
            df,
            schema_df,
            column_name_col=SCHEMA_COLUMN_NAME,
            role_col=SCHEMA_ROLE_NAME,
            feature_role_value=FEATURE_ROLE_VALUE,
        )
    else:
        feature_columns = get_feature_columns_by_drop_rule(df, drop_columns)

    check_required_targets(df, REGRESSION_TARGET, CLASSIFICATION_TARGET)
    check_leakage_columns(feature_columns, FORBIDDEN_FEATURE_KEYWORDS)

    if REQUIRE_NUMERIC_FEATURES:
        check_numeric_features(df, feature_columns)

    return feature_columns


def split_features_targets(
    df: pd.DataFrame,
    feature_columns: list[str],
    regression_target: str,
    classification_target: str,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    check_required_targets(df, regression_target, classification_target)

    x = df[feature_columns]
    y_reg = df[regression_target]
    y_cls = df[classification_target]
    return x, y_reg, y_cls


# 실제 실행 예시
# feature_columns = get_feature_columns(df, DROP_COLUMNS, SCHEMA_PATH)
# x, y_reg, y_cls = split_features_targets(df, feature_columns, REGRESSION_TARGET, CLASSIFICATION_TARGET)


## 5. 시간 기준 Train / Validation Split

조류경보 예측은 시계열 문제이므로 무작위 split을 기본값으로 사용하지 않습니다.

우선순위:
1. 전처리팀이 만든 `split` 컬럼이 있으면 사용
2. 없으면 `sample_date` 기준으로 정렬 후 앞 80% train, 뒤 20% validation


In [ ]:
def time_based_train_valid_split(
    df: pd.DataFrame,
    date_column: str,
    split_column: str | None = None,
    valid_size_ratio: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if split_column and split_column in df.columns:
        train_df = df[df[split_column] == "train"].copy()
        valid_df = df[df[split_column].isin(["valid", "validation"])].copy()
        if len(train_df) == 0 or len(valid_df) == 0:
            raise ValueError("split column exists, but train/valid rows are empty. Check split labels.")
        return train_df, valid_df

    if date_column not in df.columns:
        raise KeyError(f"Date column not found: {date_column}")

    sorted_df = df.copy()
    sorted_df[date_column] = pd.to_datetime(sorted_df[date_column])
    sorted_df = sorted_df.sort_values(date_column).reset_index(drop=True)

    split_idx = int(len(sorted_df) * (1 - valid_size_ratio))
    train_df = sorted_df.iloc[:split_idx].copy()
    valid_df = sorted_df.iloc[split_idx:].copy()
    return train_df, valid_df


# 실제 실행 예시
# train_df, valid_df = time_based_train_valid_split(df, DATE_COLUMN, SPLIT_COLUMN, VALID_SIZE_RATIO)


## 6. Gradient Boosting 모델 정의

우선순위는 다음과 같습니다.

1. LightGBM
2. XGBoost
3. HistGradientBoosting

LightGBM / XGBoost가 설치되어 있지 않을 수 있으므로, 기본 fallback은 sklearn의 `HistGradientBoosting`으로 둡니다.


In [ ]:
def build_regression_model(random_state: int = 42) -> Any:
    # Priority: LightGBM -> XGBoost -> sklearn HistGradientBoosting
    try:
        from lightgbm import LGBMRegressor
        return LGBMRegressor(
            objective="regression",
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=31,
            random_state=random_state,
        )
    except ImportError:
        pass

    try:
        from xgboost import XGBRegressor
        return XGBRegressor(
            objective="reg:squarederror",
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state,
        )
    except ImportError:
        pass

    return HistGradientBoostingRegressor(
        learning_rate=0.03,
        max_iter=500,
        random_state=random_state,
    )


def build_classification_model(random_state: int = 42) -> Any:
    # Output is probability of alert-risk class.
    try:
        from lightgbm import LGBMClassifier
        return LGBMClassifier(
            objective="binary",
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=31,
            random_state=random_state,
        )
    except ImportError:
        pass

    try:
        from xgboost import XGBClassifier
        return XGBClassifier(
            objective="binary:logistic",
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=random_state,
        )
    except ImportError:
        pass

    return HistGradientBoostingClassifier(
        learning_rate=0.03,
        max_iter=500,
        random_state=random_state,
    )


## 7. 모델 학습

회귀 모델과 분류 모델을 분리해서 학습합니다.

- 회귀: 7일 뒤 또는 다음 채수일 유해남조류 세포수 관련 target
- 분류: 관심 이상 여부 target


In [ ]:
def train_models(
    train_df: pd.DataFrame,
    feature_columns: list[str],
    regression_target: str,
    classification_target: str,
    random_state: int = 42,
) -> dict[str, Any]:
    x_train = train_df[feature_columns]
    y_reg_train = train_df[regression_target]
    y_cls_train = train_df[classification_target]

    reg_model = build_regression_model(random_state=random_state)
    cls_model = build_classification_model(random_state=random_state)

    reg_model.fit(x_train, y_reg_train)
    cls_model.fit(x_train, y_cls_train)

    return {
        "regression_model": reg_model,
        "classification_model": cls_model,
        "feature_columns": feature_columns,
    }


# 실제 실행 예시
# feature_columns = get_feature_columns(train_df, DROP_COLUMNS)
# trained = train_models(train_df, feature_columns, REGRESSION_TARGET, CLASSIFICATION_TARGET, RANDOM_STATE)


## 8. 모델 평가 코드

성능 수치는 실제 데이터를 실행한 뒤 확인합니다.

분류 문제에서는 위험 미탐을 줄이는 것이 중요하므로 `Recall`을 함께 확인합니다.


In [ ]:
def inverse_log10_cells(pred_log_cells: np.ndarray) -> np.ndarray:
    # NOTE:
    # 아래 복원식은 target이 log10(cells + 1)로 생성된 경우에만 사용합니다.
    # target 변환 방식이 다르면 inverse transform 함수를 수정해야 합니다.
    return (10 ** pred_log_cells) - 1


def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse,
        "r2": float(r2_score(y_true, y_pred)),
    }


def classification_metrics(
    y_true: pd.Series,
    risk_probability: np.ndarray,
    probability_threshold: float = 0.5,
) -> dict[str, Any]:
    y_pred = (risk_probability >= probability_threshold).astype(int)

    metrics = {
        "threshold": probability_threshold,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }

    if len(pd.Series(y_true).dropna().unique()) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, risk_probability))
    else:
        metrics["roc_auc"] = None

    return metrics


def evaluate_models(
    trained: dict[str, Any],
    valid_df: pd.DataFrame,
    regression_target: str,
    classification_target: str,
    probability_threshold: float = 0.5,
) -> dict[str, Any]:
    feature_columns = trained["feature_columns"]
    x_valid = valid_df[feature_columns]

    reg_pred = trained["regression_model"].predict(x_valid)
    cls_proba = trained["classification_model"].predict_proba(x_valid)[:, 1]

    return {
        "regression": regression_metrics(valid_df[regression_target], reg_pred),
        "classification": classification_metrics(valid_df[classification_target], cls_proba, probability_threshold),
    }


# 실제 실행 예시
# metrics = evaluate_models(trained, valid_df, REGRESSION_TARGET, CLASSIFICATION_TARGET, PROBABILITY_THRESHOLD)
# metrics


## 9. 모델 저장 / 로드


In [ ]:
def save_models(trained: dict[str, Any], output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(trained["regression_model"], output_dir / "regression_model.joblib")
    joblib.dump(trained["classification_model"], output_dir / "classification_model.joblib")

    metadata = {
        "feature_columns": trained["feature_columns"],
        "regression_target": REGRESSION_TARGET,
        "classification_target": CLASSIFICATION_TARGET,
        "probability_threshold": PROBABILITY_THRESHOLD,
        "alert_cell_threshold": ALERT_CELL_THRESHOLD,
        "regression_target_is_log10_cells_plus_1": REGRESSION_TARGET_IS_LOG10_CELLS_PLUS_1,
    }
    with open(output_dir / "model_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)


def load_models(model_dir: Path) -> dict[str, Any]:
    regression_model = joblib.load(model_dir / "regression_model.joblib")
    classification_model = joblib.load(model_dir / "classification_model.joblib")

    with open(model_dir / "model_metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

    return {
        "regression_model": regression_model,
        "classification_model": classification_model,
        "feature_columns": metadata["feature_columns"],
        "metadata": metadata,
    }


# 실제 실행 예시
# save_models(trained, OUTPUT_DIR)
# loaded = load_models(OUTPUT_DIR)


## 10. 추론 함수

입력은 이미 전처리가 끝난 모델 입력 테이블이어야 합니다.

이 함수에서도 새로운 feature를 만들지 않습니다.


In [ ]:
def predict_with_models(
    trained: dict[str, Any],
    input_df: pd.DataFrame,
    restore_cells: bool = True,
) -> pd.DataFrame:
    feature_columns = trained["feature_columns"]
    x = input_df[feature_columns]

    pred_reg = trained["regression_model"].predict(x)
    pred_proba = trained["classification_model"].predict_proba(x)[:, 1]

    output = input_df[[col for col in [DATE_COLUMN, SITE_COLUMN] if col in input_df.columns]].copy()
    output["pred_regression_target"] = pred_reg

    if restore_cells and REGRESSION_TARGET_IS_LOG10_CELLS_PLUS_1:
        # NOTE:
        # 아래 복원식은 target이 log10(cells + 1)로 생성된 경우에만 사용합니다.
        # target 변환 방식이 다르면 inverse transform 함수를 수정해야 합니다.
        output["pred_cells"] = inverse_log10_cells(pred_reg)

    output["alert_risk_probability"] = pred_proba
    output["alert_pred_label"] = (pred_proba >= PROBABILITY_THRESHOLD).astype(int)
    return output


# 실제 실행 예시
# prediction_df = predict_with_models(loaded, df)
# prediction_df.head()


## 11. 조류경보 운영 보조 로직

이 로직은 공식 발령을 자동 결정하지 않습니다.

예측 결과를 바탕으로 관리자가 사전 점검할 수 있는 `관심 단계 후보`를 표시하는 보조 함수입니다.


In [ ]:
def apply_operational_alert_logic(
    prediction_df: pd.DataFrame,
    previous_cells_column: str = "previous_observed_cells",  # TODO: 실제 컬럼명에 맞게 수정
    pred_cells_column: str = "pred_cells",
    probability_column: str = "alert_risk_probability",
    alert_cell_threshold: int = 1000,
    probability_threshold: float = 0.5,
) -> pd.DataFrame:
    # NOTE:
    # 이 함수는 조류경보 공식 발령을 자동 결정하는 코드가 아닙니다.
    # 예측 결과를 바탕으로 관리자가 사전 점검할 수 있는
    # '관심 단계 후보'를 표시하는 운영 보조 로직입니다.
    output = prediction_df.copy()

    has_previous_cells = previous_cells_column in output.columns
    has_pred_cells = pred_cells_column in output.columns
    has_probability = probability_column in output.columns

    previous_exceeded = output[previous_cells_column] >= alert_cell_threshold if has_previous_cells else False
    predicted_cells_exceeded = output[pred_cells_column] >= alert_cell_threshold if has_pred_cells else False
    predicted_probability_exceeded = output[probability_column] >= probability_threshold if has_probability else False

    output["alert_candidate"] = previous_exceeded & (predicted_cells_exceeded | predicted_probability_exceeded)
    return output


# 실제 실행 예시
# operational_df = apply_operational_alert_logic(prediction_df)
# operational_df.head()


## 12. 설명 함수: SHAP 또는 Feature Importance

SHAP이 설치되어 있으면 SHAP 기반 summary table을 만들 수 있습니다.

설치되어 있지 않거나 모델 호환 문제가 있으면 feature importance 기반 설명을 사용합니다.

실제 SHAP 값과 중요도 결과는 데이터를 실행한 뒤 확인해야 합니다.


In [ ]:
def get_feature_importance_table(model: Any, feature_columns: list[str]) -> pd.DataFrame:
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
        return (
            pd.DataFrame({"feature": feature_columns, "importance": importance})
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )

    return pd.DataFrame(columns=["feature", "importance"])


def make_shap_summary_table(
    trained: dict[str, Any],
    input_df: pd.DataFrame,
    prediction_df: pd.DataFrame,
    model_key: str = "classification_model",
    top_n: int = 3,
) -> pd.DataFrame:
    # Return top SHAP features per row as a summary table.
    # Actual SHAP values must be computed by running this code with real data.
    try:
        import shap
    except ImportError as exc:
        raise ImportError("shap is not installed. Install shap or use feature importance instead.") from exc

    feature_columns = trained["feature_columns"]
    model = trained[model_key]
    x = input_df[feature_columns]

    explainer = shap.Explainer(model, x)
    shap_values = explainer(x)

    rows = []
    for row_idx in range(len(x)):
        values = np.asarray(shap_values.values[row_idx])
        top_indices = np.argsort(np.abs(values))[::-1][:top_n]

        row = {}
        for col in [DATE_COLUMN, SITE_COLUMN]:
            if col in input_df.columns:
                row[col] = input_df.iloc[row_idx][col]

        if "alert_risk_probability" in prediction_df.columns:
            row["risk_probability"] = prediction_df.iloc[row_idx]["alert_risk_probability"]
        if "pred_cells" in prediction_df.columns:
            row["prediction"] = prediction_df.iloc[row_idx]["pred_cells"]

        for rank, feature_idx in enumerate(top_indices, start=1):
            row[f"top_{rank}_feature"] = feature_columns[feature_idx]
            row[f"top_{rank}_shap_value"] = values[feature_idx]
        rows.append(row)

    return pd.DataFrame(rows)


# 실제 실행 예시
# importance_df = get_feature_importance_table(loaded["classification_model"], loaded["feature_columns"])
# shap_summary_df = make_shap_summary_table(loaded, df, prediction_df)


## 13. 전체 실행 순서 예시

아래 셀은 전체 흐름을 한 번에 보여주기 위한 예시입니다.

실제 실행 전 config의 컬럼명을 전처리팀 산출물에 맞게 수정해야 합니다.


In [ ]:
# df = load_model_input(MODEL_INPUT_PATH)
# train_df, valid_df = time_based_train_valid_split(df, DATE_COLUMN, SPLIT_COLUMN, VALID_SIZE_RATIO)
# feature_columns = get_feature_columns(train_df, DROP_COLUMNS, SCHEMA_PATH)
# trained = train_models(train_df, feature_columns, REGRESSION_TARGET, CLASSIFICATION_TARGET, RANDOM_STATE)
# metrics = evaluate_models(trained, valid_df, REGRESSION_TARGET, CLASSIFICATION_TARGET, PROBABILITY_THRESHOLD)
# save_models(trained, OUTPUT_DIR)
# loaded = load_models(OUTPUT_DIR)
# prediction_df = predict_with_models(loaded, valid_df)
# operational_df = apply_operational_alert_logic(prediction_df)
# metrics


## 확인된 내용 / 추정인 내용 / 추가 확인 필요

확인된 내용:
- 최종 모델은 Gradient Boosting 계열을 고려합니다.
- 예측 출력은 회귀와 분류를 함께 고려합니다.
- Granger / VAR는 lag 후보 탐색용으로 분리해야 합니다.
- 전처리와 feature engineering은 별도 파트가 담당합니다.

추정인 내용:
- target column 이름은 전처리팀 산출물에 따라 달라질 수 있습니다.
- log10(cells + 1) 변환 여부는 target 생성 방식 확인이 필요합니다.
- 관심 기준 1,000 cells/mL는 공식 기준 확인 후 config에서 확정해야 합니다.

추가 확인이 필요한 내용:
- 실제 모델 입력 테이블 경로
- feature column 목록
- target column 이름
- site column 이름
- sample_date column 이름
- 회귀 target 변환 방식
- 분류 target 기준
- probability threshold
